# Base Conv-SNN Encoding Comparison

Compare three base models trained with `rate`, `latency`, and `delta` input encoding.

Expected run dirs:
- `runs/scnn/base_rate`
- `runs/scnn/base_latency`
- `runs/scnn/base_delta`


In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.image as mpimg

plt.rcParams['figure.figsize'] = (12, 4)


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'runs').exists():
    for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (p / 'runs').exists():
            PROJECT_ROOT = p
            break

RUNS_ROOT = PROJECT_ROOT / 'runs' / 'scnn'
SUMMARY_PATH = RUNS_ROOT / 'base_encoding_comparison' / 'summary.json'

if SUMMARY_PATH.exists():
    payload = json.loads(SUMMARY_PATH.read_text())
    run_entries = payload['runs']
else:
    run_entries = [
        {'encoding': 'rate', 'run_dir': str(RUNS_ROOT / 'base_rate')},
        {'encoding': 'latency', 'run_dir': str(RUNS_ROOT / 'base_latency')},
        {'encoding': 'delta', 'run_dir': str(RUNS_ROOT / 'base_delta')},
    ]

run_entries


In [ ]:
records = []
histories = {}

for entry in run_entries:
    enc = entry['encoding']
    run_dir = Path(entry['run_dir'])
    history_path = run_dir / 'history.json'
    metrics_path = run_dir / 'metrics.txt'

    if not history_path.exists() or not metrics_path.exists():
        print(f'[missing] {enc}: {run_dir}')
        continue

    history = json.loads(history_path.read_text())
    histories[enc] = history

    metrics = {}
    for line in metrics_path.read_text().splitlines():
        if ':' not in line:
            continue
        k, v = line.split(':', 1)
        k = k.strip()
        v = v.strip()
        try:
            metrics[k] = float(v)
        except ValueError:
            metrics[k] = v

    records.append({
        'encoding': enc,
        'run_dir': str(run_dir),
        **metrics,
    })

df = pd.DataFrame(records).sort_values('encoding').reset_index(drop=True)
df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

for enc, hist in histories.items():
    epochs = np.arange(1, len(hist.get('train_loss', [])) + 1)
    if len(epochs) == 0:
        continue
    axes[0].plot(epochs, hist['train_loss'], label=f'{enc}-train')
    axes[0].plot(epochs, hist['val_loss'], '--', label=f'{enc}-val')

    axes[1].plot(epochs, hist['train_acc'], label=f'{enc}-train')
    axes[1].plot(epochs, hist['val_acc'], '--', label=f'{enc}-val')

    axes[2].plot(epochs, hist['train_spike_count'], label=f'{enc}-train')
    axes[2].plot(epochs, hist['val_spike_count'], '--', label=f'{enc}-val')

axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('Accuracy Curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)

axes[2].set_title('Avg Output Spike Count')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Spikes / sample')
axes[2].legend(fontsize=8)

plt.show()


In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    axes[0].bar(df['encoding'], df['test_acc'])
    axes[0].set_title('Test Accuracy')
    axes[0].set_ylim(0, 1)

    axes[1].bar(df['encoding'], df['test_loss'])
    axes[1].set_title('Test Loss')

    axes[2].bar(df['encoding'], df['test_spike_count'])
    axes[2].set_title('Test Spike Count')

    plt.show()
else:
    print('No completed runs found yet.')


In [ ]:
# Display confusion matrices from each run (if present)
available = []
for entry in run_entries:
    enc = entry['encoding']
    img_path = Path(entry['run_dir']) / 'confusion_matrix.png'
    if img_path.exists():
        available.append((enc, img_path))

if not available:
    print('No confusion matrix images found yet.')
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), constrained_layout=True)
    if n == 1:
        axes = [axes]

    for ax, (enc, img_path) in zip(axes, available):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.set_title(f'{enc} confusion matrix')
        ax.axis('off')

    plt.show()
